In [51]:
import numpy as np
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split


In [52]:
with open('tokenizers/wiki_token.pkl', 'rb') as f:
    wiki_data = pickle.load(f)

print(wiki_data[0])
print(len(wiki_data))

[354, 696, 76, 11404, 2821, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13]
243883


In [53]:

# tokens and curriculum score come in, tensors of x and y go out and sample ids with difficulty
class DataLoader:
    def __init__(self, x, y, batch_size, max_len=100, pad_token=0):
        self.x, self.y = x, y
        self.batch_size = batch_size
        self.max_len = max_len
        self.pad_token = pad_token


    def PadSeq(self, seq):
        if len(seq) > self.max_len:
            return seq[:self.max_len]
        else:
            return seq + [self.pad_token] * (self.max_len - len(seq)) # pad
        
    def __iter__(self):
        perm = np.random.permutation(len(self.x))

        for i in range(0, len(self.x), self.batch_size):
            ids = perm[i : i + self.batch_size]
            batch_x = [self.PadSeq(self.x[j]) for j in ids] # target
            batch_y = [self.PadSeq(self.y[j]) for j in ids] # label

        yield torch.tensor(batch_x), torch.tensor(batch_y) # batch shape [batch_size, max_len]
        

with open('tokenizers/wiki_token.pkl', 'rb') as f:
    data = pickle.load(f)

print(data[0:5])
print(len(data))

X_train, X_test, y_train, y_test = train_test_split(data[:-1], data[1:], test_size=0.2, random_state=88)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=88)






[[354, 696, 76, 11404, 2821, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13], [17006, 570, 261, 4244, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13], [354, 559, 487, 454, 12, 7278, 12, 2213, 29658, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13], [2395, 457, 391, 4244, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13], [49916, 1236, 274, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13]]
243883


In [54]:
# ==== Configuration ==== #
@dataclass
class ModelConfig:
    def __init__(self):
        self.block_size = 8 # context size (min num of tokens in sequence, how long model can see at once)
        self.vocab_size = 10000
        self.max_iters = 100
        self.n_head = 4
        self.n_embd = 32
        self.n_layers = 4
        self.dropout = 0.
        self.bias: bool = True
        self.device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
                                          # switch out when transferring to github or stanage for cuda use

@dataclass
class TrainConfig:
    epochs: int = 150
    batch_size: int = 4
    max_len: int = 100
    pad: int = 0
    grad_accum_steps: int = 1
    max_lr: float = 3e-4
    min_lr: float = 1e-5
    warmup_iters: int = 100
    lr_decay_iters: int = 1000
    save_every: int = 1
    # add optimiser hp


# GPT2 Decoder #

Components:

1. Token Embeddings - Map token ids to dense vecs

2. Positional Encoding - add positional info to embeddings

3. Transformer Blocks (n) - self-attention + feed foward layers

4. Casual masking - prevents peeking into future

5. Final Linear Layer - projects to vocab size (logits)

In [ ]:
# ==== GPT2 Decoder ==== #

### UPDATE FOR TORCH USE ###

class GPT2Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.config = config

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias) # create W attn embds
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        # layer norm
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias), # expand dims (4 * from Attention all u need paper)
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd), # project back 
            nn.Dropout(config.dropout)
        )
        
        self.attn_dropout = nn.Dropout(config.dropout)


    def SelfAttention(self,x, mask=None):
        b,t,c = x.shape
        q, k, v = self.c_attn(x).split(self.config.n_embd, dim=2)
        
        q =q.reshape(b, t, self.config.n_head, c//self.config.n_head).permute(0,2,1,3)
        k = k.reshape(b, t, self.config.n_head, c//self.config.n_head).permute(0,2,1,3)
        v = v.reshape(b, t, self.config.n_head, c//self.config.n_head).permute(0,2,1,3)

        score = (q @ k.transpose(-2,-1)) / math.sqrt(k.shape[-1])

        if mask is not None:
            score = score + mask

        a = nn.functional.softmax(score, dim=-1)
        a = self.attn_dropout(a)
        head = a @ v
        out = head.permute(0,2,1,3).reshape(b,t,c) # back to input dims
        out = self.c_proj(out)
        #print("Type of self.SelfAttention(x):", type(out))
        return out
    
    def forward(self, x, mask=None):
        # self attention ad layer norm
        x = x + self.SelfAttention(self.ln1(x), mask)
        x = x + self.mlp(self.ln2(x)) # resudual connections
       
        return x
    

class GPT2Model(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embd = nn.Embedding(config.vocab_size, config.n_embd)
        self.posn_embd = nn.Embedding(config.block_size, config.n_embd)

        self.proj = nn.Linear(config.n_embd, config.vocab_size)

        self.blocks = nn.Sequential(*[GPT2Block(config) for _ in range(config.n_layers)])
        self.fln = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        b, t = x.size()
        tokn_embd = self.token_embd(x)
        posn_idcs = torch.arange(t, device=x.device).unsqueeze(0).expand(b,t)
        posn_embd = self.posn_embd(posn_idcs)

        x = tokn_embd + posn_embd

        x = self.blocks(x)
        x = self.fln(x) # final linear layer
        return self.proj(x) # project back to (enbedding size, vocab size)


class GPTTrainer: # ADD Validation and test 
    def __init__(self, x, y, train_config, model_config, check_points_dir=None):
        self.x, self.y = x, y
        self.train_config = train_config
        self.check_points_dir = check_points_dir

        self.device = model_config.device

        self.data_loader = DataLoader(
                                    self.x, self.y, 
                                    train_config.batch_size, 
                                    train_config.max_len,
                                    train_config.pad
                                    )
        
        # Intitialise model and optimiser
        self.model = GPT2Model(model_config).to(self.device)
        self.optim = torch.optim.AdamW(self.model.parameters(), lr=train_config.max_lr)
        self.criterion = nn.CrossEntropyLoss()


    def step(self, inputs, targets):
        self.model.train()

        inputs = inputs.to(self.device)
        targets = targets.to(self.device)

        logits = self.model(inputs) # foward pass

        b, t, v = logits.shape
        logits = logits.view(b*t, v)
        targets = targets.view(b*t)

        loss = self.criterion(logits, targets)

        # back prop
        self.optim.zero_grad()
        loss.backward()
        self.optim.step()

        return loss.item()

    
    def train(self):
        for epoch in range(self.train_config.epochs):
            total_loss = 0.
            n_batches = 0

            for x, y in self.data_loader:
                #print(f"Batch {n_batches+1}")
                loss_val = self.step(x, y)
                total_loss += loss_val
                n_batches +=1

            avg_loss = total_loss / n_batches
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.3f}")

            # save checpoints
            if self.check_points_dir and (epoch + 1) % self.train_config.save_every ==0:
                torch.save(f"{self.check_points_dir}/model_epoch_{epoch+1}.npz", **self.model.state_dict())

 
class GenerateGPT:
    def __init__(self, model, tokeniser, prompt, max_len=50):
        super.__init__()
 


# Text Generation # 

Predicts next token from the given context. 

1. Input [cat sat on mat] -> token representation 

2. Feed tokens to the trained model. It will output tensor with shape [batch_size, seq_len, vocab_size], which will have the probability distribution over the vocab at each position. Only care about the last position to predict next token.

3. Pick the token. From the final distribution: Argmax for highest prob, or sample from the distribution (more creative) or use top-k/top-p (nucleus) sampling to balance diversity and equity. Then simply append token to sequence.

4. Repeat, feed the model the updated sequence back in and repeat 2 and 3 until a max length is reached or model generates and end of sequence token.

### Casual Masking ###
In training and generation, model cant peak into the future.

Tests:
- Different prompt lengths
- Adjust temperature (controls randomness in sampling)
- Add stop conditions (e.g if it generates a period or newline)
- see how model behaves w/ different types of prompt

In [56]:

#loader = DataLoader(x, y, batch_size=8, max_len=100, pad_token=0)


model_config = ModelConfig()
train_config = TrainConfig()


trainGPT = GPTTrainer(X_train, y_train, train_config, model_config, check_points_dir=None)
trainGPT.train()

Epoch 1 | Loss: 9.003
Epoch 2 | Loss: 8.426
Epoch 3 | Loss: 8.601
Epoch 4 | Loss: 8.024
Epoch 5 | Loss: 8.045
Epoch 6 | Loss: 8.385
Epoch 7 | Loss: 8.685
Epoch 8 | Loss: 8.544
Epoch 9 | Loss: 8.313
Epoch 10 | Loss: 8.568
Epoch 11 | Loss: 8.171
Epoch 12 | Loss: 8.002
Epoch 13 | Loss: 8.399
Epoch 14 | Loss: 8.258
Epoch 15 | Loss: 8.135
Epoch 16 | Loss: 7.725
Epoch 17 | Loss: 8.308
Epoch 18 | Loss: 8.154
Epoch 19 | Loss: 8.242
Epoch 20 | Loss: 8.265
Epoch 21 | Loss: 7.830
Epoch 22 | Loss: 8.220
Epoch 23 | Loss: 7.783
Epoch 24 | Loss: 8.298
Epoch 25 | Loss: 7.967
Epoch 26 | Loss: 7.905
Epoch 27 | Loss: 7.888
Epoch 28 | Loss: 7.785
Epoch 29 | Loss: 7.592
Epoch 30 | Loss: 7.445
Epoch 31 | Loss: 8.053
Epoch 32 | Loss: 7.500
Epoch 33 | Loss: 7.579
Epoch 34 | Loss: 7.574
Epoch 35 | Loss: 7.333
Epoch 36 | Loss: 7.885
Epoch 37 | Loss: 7.549
Epoch 38 | Loss: 7.761
Epoch 39 | Loss: 7.488
Epoch 40 | Loss: 7.717
Epoch 41 | Loss: 7.608
Epoch 42 | Loss: 7.250
Epoch 43 | Loss: 7.475
Epoch 44 | Loss: 7.8